In [ ]:
from huggingface_hub import login

login("KEY")

import os
import json
import time

import pandas as pd
import whisperx
import torchaudio


# =========================
# CONFIGURACIÓN
# =========================

carpeta_audios = "/content/drive/MyDrive/Audios_Issabel"
carpeta_json = "/content/drive/MyDrive/Transcripciones_diarización"
salida_excel = "/content/drive/MyDrive/resumen_transcripciones2.xlsx"

device = "cuda"

# Coloque su token en una variable de entorno o en Colab Secrets
#HF_TOKEN = os.getenv("HF_TOKEN")

os.makedirs(carpeta_json, exist_ok=True)


# =========================
# FUNCIÓN PARA SERIALIZAR JSON
# =========================

def convertir_json(obj):
    try:
        return obj.item()
    except AttributeError:
        return str(obj)


# =========================
# MODELOS
# =========================

print("⏳ Cargando WhisperX...")

model = whisperx.load_model(
    "medium",
    device,
    language="es"
)

print("⏳ Cargando modelo de alineación...")

model_a, metadata = whisperx.load_align_model(
    language_code="es",
    device=device
)

print("⏳ Cargando modelo de diarización...")

diarize_model = whisperx.diarize.DiarizationPipeline(
    device=device
)


# =========================
# RESULTADOS
# =========================

resultados_totales = []


# =========================
# RECORRER AUDIOS
# =========================

audios = [
    archivo for archivo in os.listdir(carpeta_audios)
    if archivo.endswith(".wav")
]

print(f"🎧 Audios encontrados: {len(audios)}")


for archivo in audios[20:]:

    try:
        print("\n==============================")
        print(f"🎙 Procesando: {archivo}")

        audio_file = os.path.join(carpeta_audios, archivo)

        # =========================
        # DURACIÓN AUDIO
        # =========================

        waveform, sample_rate = torchaudio.load(audio_file)

        duracion_segundos = waveform.shape[1] / sample_rate

        duracion_minutos = round(
            duracion_segundos / 60,
            2
        )

        # =========================
        # INICIO TIEMPO
        # =========================

        inicio = time.time()

        # =========================
        # TRANSCRIPCIÓN
        # =========================

        result = model.transcribe(
            audio_file,
            language="es"
        )

        # =========================
        # ALINEACIÓN TEMPORAL
        # =========================

        result = whisperx.align(
            result["segments"],
            model_a,
            metadata,
            audio_file,
            device
        )

        # =========================
        # DIARIZACIÓN
        # =========================

        diarize_segments = diarize_model(
            audio_file,
            min_speakers=2,
            max_speakers=2
        )

        result = whisperx.assign_word_speakers(
            diarize_segments,
            result
        )

        # =========================
        # FIN TIEMPO
        # =========================

        fin = time.time()

        tiempo_transcripcion = round(
            fin - inicio,
            2
        )

        print("✅ Transcripción + diarización completada")

        # =========================
        # GUARDAR JSON
        # =========================

        nombre_json = os.path.splitext(archivo)[0] + ".json"

        ruta_json = os.path.join(
            carpeta_json,
            nombre_json
        )

        with open(
            ruta_json,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                result,
                f,
                ensure_ascii=False,
                indent=4,
                default=convertir_json
            )

        print(f"💾 JSON guardado: {nombre_json}")

        # =========================
        # RESUMEN EXCEL
        # =========================

        resultados_totales.append({
            "archivo": archivo,
            "duracion_minutos": duracion_minutos,
            "tiempo_transcripcion_segundos": tiempo_transcripcion
        })

    except Exception as e:

        print(f"❌ Error procesando {archivo}")
        print(e)


# =========================
# EXPORTAR EXCEL
# =========================

resultado_df = pd.DataFrame(resultados_totales)

resultado_df.to_excel(
    salida_excel,
    index=False
)

print("\n===================================")
print("✅ PROCESO FINALIZADO")
print(f"📁 Excel guardado en: {salida_excel}")